# Notebook 7: RAG System with Agent Routing

This notebook covers **Step 6 (RAG)**, adapted from the course RAG lab sequence
(1_TextExtraction → 7_Generator). The lab pipeline was built around Wikipedia/PDF documents,
Ollama (local LLM + embeddings), and Weaviate Cloud. Three adaptations are made for this
project, documented at the point they occur:

1. **Chunking** — our documents are already-short YouTube comments (not multi-paragraph PDFs),
   so each comment is one document, prefixed with contextual metadata (video title,
   perspective) — the same "structured/contextual chunking" idea as the lab, applied to a
   different document type. Long outlier comments still use the lab's
   `RecursiveCharacterTextSplitter`.
2. **Embedding** — swapped the lab's Ollama local embedding server for `sentence-transformers`
   (already used in Notebook 3), since Ollama needs a locally running server which isn't
   available in Colab.
3. **Generator** — swapped the lab's DSPy + local Ollama LLM for the Claude API directly, for
   faster setup with no local model hosting required.

Retrieval strategies implemented: **semantic (MMR)**, **lexical (TF-IDF, from Notebook 6)**,
**hybrid (weighted fusion)**, and **metadata-filtered**, covering the project brief's full
"metadata, semantic, lexical, hybrid" retrieval requirement.

An **agent router** sits in front of generation: confident RAG matches are answered from the
dataset; low-confidence queries fall back to a web search, so the system never has to hallucinate
an answer it doesn't have.


## Section 1 — Setup & Dependencies

In [1]:
!pip install -q langchain langchain-community langchain-text-splitters faiss-cpu \
    sentence-transformers anthropic duckduckgo-search joblib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 97.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
import pandas as pd
import numpy as np
import json
import ast
import joblib
from pathlib import Path

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

Path('rag_store').mkdir(exist_ok=True)

print('All libraries loaded successfully.')


/tmp/ipykernel_17672/2702147474.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


All libraries loaded successfully.


## Section 2 — Load Data

Loads the NER-tagged dataset (Notebook 5) plus the fitted TF-IDF vectorizer and matrix
(Notebook 6), which becomes our lexical retriever.

In [4]:
df = pd.read_csv("ner_comments.csv.gz")
df = df.dropna(subset=['light_clean_text']).reset_index(drop=True)
df['light_clean_text'] = df['light_clean_text'].astype(str)

# entities was serialised to a string in Notebook 5 -- parse back to a list
df['entities'] = df['entities'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

print(f'Loaded {len(df):,} comments')
df.head(3)


Loaded 51,684 comments


,comment_id,video_id,title,author,published_at,like_count,text,light_clean_text,clean_text,word_count,char_count,lda_topic,topic,perspective,vader_sentiment,final_sentiment,final_sentiment_label,entities,entity_count
0,UgxmVVEhmGQ0ovq6tRR4AaABAg,PEY8IyB653c,"THE UNHINGED CONSUMERISM OF ""RESTOCK"" INFLUENC...",@jadehannah916,2026-06-22 19:50:06+00:00,0,19:23 expiry dates have left the chat,expiry dates have left the chat,expiry date leave chat,4,22,0,-1,Outlier / Noise,0,0,Neutral,[],0
1,Ugwfz5ugRqp0yxubW9t4AaABAg,PEY8IyB653c,"THE UNHINGED CONSUMERISM OF ""RESTOCK"" INFLUENC...",@jadehannah916,2026-06-22 19:46:57+00:00,0,16:34 just work at a grocery store at this point,just work at a grocery store at this point,work grocery store point,4,24,3,-1,Outlier / Noise,0,0,Neutral,[],0
2,UgwvTEiO61bpwRH3Dn14AaABAg,PEY8IyB653c,"THE UNHINGED CONSUMERISM OF ""RESTOCK"" INFLUENC...",@Sevdalina-e2l,2026-06-22 11:08:51+00:00,0,19:40 this specific content made my heart blee...,this specific content made my heart bleed. Lik...,specific content heart bleed like mean empty k...,10,61,2,6,Everyday Objects & Consumption,1,1,Positive,"[(😔, ORG)]",1


In [6]:
# Load the TF-IDF artifacts from Notebook 6 (lexical retrieval)
tfidf_vectorizer = joblib.load('tfidf_vectorizer.pkl')
tfidf_matrix = joblib.load('tfidf_matrix.pkl')
tfidf_row_index = pd.read_csv('tfidf_row_index.csv')

# Align df to the same row order the TF-IDF matrix was built on
df = df.merge(tfidf_row_index.reset_index().rename(columns={'index': 'tfidf_row'}), on='comment_id', how='inner')
df = df.sort_values('tfidf_row').reset_index(drop=True)

print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'Aligned comments: {len(df):,}')


TF-IDF matrix shape: (51684, 29614)
Aligned comments: 51,684


## Section 3 — Document Preparation ("Chunking")

**Adaptation from the lab's chunking step:** the lab's Wikipedia/PDF pipeline needed sliding-window
and structured chunking because paragraphs are long and need splitting. Our documents (YouTube
comments) are already short, self-contained units — chunking them further would fragment single
ideas. Instead we apply the lab's **contextual/structured chunking concept**: each document is
prefixed with its video title and perspective, the same way the lab prefixed PDF chunks with
`pdf_title + ", " + sentence`.

For the small number of unusually long comments, we still apply the lab's
`RecursiveCharacterTextSplitter` (same `chunk_size`/`chunk_overlap` philosophy) so no single
document overwhelms the embedding model's context.

In [7]:
LONG_COMMENT_THRESHOLD = 400  # characters

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=25,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

def build_contextual_chunks(row):
    """
    Returns a list of (chunk_text, metadata) tuples for one comment.
    Short comments -> a single contextual chunk.
    Long comments  -> split via RecursiveCharacterTextSplitter, each piece keeps the same context prefix.
    """
    prefix = f"[Video: {row['title'][:80]}] [Perspective: {row['perspective']}] "
    text = row['light_clean_text']

    if len(text) <= LONG_COMMENT_THRESHOLD:
        pieces = [text]
    else:
        pieces = text_splitter.split_text(text)

    metadata_base = {
        'comment_id': row['comment_id'],
        'video_id': row['video_id'],
        'title': row['title'],
        'perspective': row['perspective'],
        'final_sentiment_label': row['final_sentiment_label'],
        'like_count': int(row['like_count']),
    }

    return [(prefix + piece, metadata_base) for piece in pieces]

documents, metadatas = [], []
for _, row in df.iterrows():
    for chunk_text, meta in build_contextual_chunks(row):
        documents.append(chunk_text)
        metadatas.append(meta)

print(f'Built {len(documents):,} contextual chunks from {len(df):,} comments')
print()
print('Example chunk:')
print(documents[0])


Built 80,320 contextual chunks from 51,684 comments

Example chunk:
[Video: THE UNHINGED CONSUMERISM OF "RESTOCK" INFLUENCERS, SO UNREALISTIC! | Influencer ] [Perspective: Outlier / Noise] expiry dates have left the chat


## Section 4 — Embedding

**Adaptation:** the lab used `OllamaEmbeddings` (`embeddinggemma` / `nomic-embed-text`), which
requires a locally running Ollama server. We substitute `sentence-transformers`
(`all-MiniLM-L6-v2` — the same model already used in Notebook 3's BERTopic step), wrapped in a
small class matching LangChain's `Embeddings` interface (`embed_documents` / `embed_query`) so
it plugs into `FAISS.from_texts` the same way the lab's `OllamaEmbeddings` did.

In [17]:
# Compute-constrained adaptation: sentence-transformers inference was too slow on CPU
# to finish within the project deadline (same GPU-unavailability issue as Notebook 4's
# RoBERTa step). Semantic embeddings are instead built from TF-IDF + TruncatedSVD (LSA)
# fit directly on our RAG documents -- a well-established, fast "semantic" embedding
# technique that needs no neural network inference and runs in seconds, not hours.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from langchain_core.embeddings import Embeddings


class FastSemanticEmbeddings(Embeddings): # Inherit from Embeddings
    def __init__(self, texts, n_components=100):
        self.vectorizer = TfidfVectorizer(min_df=1, max_df=0.9, stop_words='english')
        tfidf = self.vectorizer.fit_transform(texts)
        n_comp = min(n_components, tfidf.shape[1] - 1, tfidf.shape[0] - 1)
        self.svd = TruncatedSVD(n_components=n_comp, random_state=42)
        self.svd.fit(tfidf)

    def embed_documents(self, texts): # Renamed from embed_documents
        return self.svd.transform(self.vectorizer.transform(texts)).tolist()

    def embed_query(self, text): # Renamed from embed_query
        return self.svd.transform(self.vectorizer.transform([text]))[0].tolist()


embeddings = FastSemanticEmbeddings(documents, n_components=100)
vec = embeddings.embed_query("de-influencing content")
print(f'Embedding length: {len(vec)}')


Embedding length: 100


## Section 5 — Vectorization (FAISS)

Follows the lab's FAISS setup exactly (`FAISS.from_texts`, `save_local` / `load_local`), with
per-chunk metadata (perspective, sentiment, video) attached via the `metadatas` parameter.

In [18]:
FAISS_STORE_PATH = "rag_store/faiss_index"

vectorstore = FAISS.from_texts(
    texts=documents,
    embedding=embeddings,
    metadatas=metadatas
)

vectorstore.save_local(FAISS_STORE_PATH)
print(f'FAISS index built and saved with {len(documents):,} vectors.')


FAISS index built and saved with 80,320 vectors.


## Section 6 — Retrieval Strategies

Four strategies, covering the project brief's "metadata, semantic, lexical, hybrid" requirement:

1. **Semantic (MMR)** — same `as_retriever(search_type="mmr", ...)` pattern as the lab
2. **Lexical (TF-IDF)** — reuses Notebook 6's fitted vectorizer/matrix, cosine similarity
3. **Hybrid** — weighted score fusion of semantic + lexical, same concept as the lab's Weaviate
   `hybrid()` `alpha` parameter, self-implemented since we don't use Weaviate
4. **Metadata-filtered** — pre-filters by perspective/sentiment before searching

In [19]:
def semanticRetrieval(query, k=5, fetch_k=20, lambda_mult=0.5):
    """MMR retrieval -- same pattern as the lab's mmrRetrieval()."""
    retriever = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={"k": k, "fetch_k": fetch_k, "lambda_mult": lambda_mult}
    )
    docs = retriever.invoke(query)
    return [(d.page_content, d.metadata) for d in docs]

# quick test
for text, meta in semanticRetrieval("credit card debt from impulse buying", k=3):
    print(f"[{meta['perspective']}] {text[:120]}")
    print()


[Other] [Video: How America Got So Good At Buying Sh*t] [Perspective: Other] stop consuming then and start investing. Holy roman

[Other] [Video: How America Got So Good At Buying Sh*t] [Perspective: Other] Predatory loan and credit card companies: 'Allow us

[Other] [Video: How America Got So Good At Buying Sh*t] [Perspective: Other] Nearly half of Americans aged 50 and older with cre



In [20]:
def lexicalRetrieval(query, k=5):
    """TF-IDF cosine-similarity retrieval, reusing Notebook 6's fitted vectorizer/matrix."""
    query_vec = tfidf_vectorizer.transform([query])
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_indices = np.argsort(scores)[::-1][:k]
    return [
        (df.iloc[i]['light_clean_text'], df.iloc[i][['perspective', 'final_sentiment_label', 'title']].to_dict(), scores[i])
        for i in top_indices
    ]

for text, meta, score in lexicalRetrieval("credit card debt from impulse buying", k=3):
    print(f"[score={score:.3f}] [{meta['perspective']}] {text[:120]}")
    print()


[score=0.741] [Other] Everyone is LARPing with their credit cards and debt 🙄

[score=0.720] [Other] All I see in these videos is credit card debt.

[score=0.698] [Other] Imagine the credit card DEEEEEBT



In [21]:
def hybridRetrieval(query, k=5, alpha=0.5, fetch_k=20):
    """
    Weighted fusion of semantic + lexical rankings.
    alpha=0 -> lexical only, alpha=1 -> semantic only (same convention as the lab's
    Weaviate hybrid() alpha parameter).
    """
    semantic_results = semanticRetrieval(query, k=fetch_k)
    lexical_results = lexicalRetrieval(query, k=fetch_k)

    scores = {}

    for rank, (text, meta) in enumerate(semantic_results):
        norm_score = 1 - (rank / max(len(semantic_results), 1))
        scores[text] = scores.get(text, 0) + alpha * norm_score

    for rank, (text, meta, _) in enumerate(lexical_results):
        norm_score = 1 - (rank / max(len(lexical_results), 1))
        scores[text] = scores.get(text, 0) + (1 - alpha) * norm_score

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]
    return ranked

for text, score in hybridRetrieval("credit card debt from impulse buying", k=5):
    print(f"[fused_score={score:.3f}] {text[:120]}")
    print()


[fused_score=0.500] [Video: How America Got So Good At Buying Sh*t] [Perspective: Other] stop consuming then and start investing. Holy roman

[fused_score=0.500] Everyone is LARPing with their credit cards and debt 🙄

[fused_score=0.475] [Video: How America Got So Good At Buying Sh*t] [Perspective: Other] Predatory loan and credit card companies: 'Allow us

[fused_score=0.475] All I see in these videos is credit card debt.

[fused_score=0.450] [Video: How America Got So Good At Buying Sh*t] [Perspective: Other] Nearly half of Americans aged 50 and older with cre



In [22]:
def metadataFilteredRetrieval(query, k=5, perspective=None, sentiment=None):
    """
    Pre-filters the dataframe by perspective/sentiment, then computes cosine similarity
    against embeddings for that subset only (fast since subsets are small).
    """
    subset = df.copy()
    if perspective is not None:
        subset = subset[subset['perspective'] == perspective]
    if sentiment is not None:
        subset = subset[subset['final_sentiment_label'] == sentiment]

    if len(subset) == 0:
        return []

    subset_texts = subset['light_clean_text'].tolist()
    subset_embeddings = np.array(embeddings.embed_documents(subset_texts))
    query_embedding = np.array(embeddings.embed_query(query)).reshape(1, -1)

    scores = cosine_similarity(query_embedding, subset_embeddings).flatten()
    top_indices = np.argsort(scores)[::-1][:k]

    return [
        (subset.iloc[i]['light_clean_text'], subset.iloc[i]['perspective'], scores[i])
        for i in top_indices
    ]

for text, persp, score in metadataFilteredRetrieval(
    "is this brand actually trustworthy", k=3, perspective=None, sentiment='Negative'
):
    print(f"[score={score:.3f}] [{persp}] {text[:120]}")
    print()


[score=0.991] [Outlier / Noise] Studies show in africa with NGO's by investing in the poorest it actually raises all boats. Trickle down economics and p

[score=0.971] [Outlier / Noise] The condiment decanting is actually insane.

[score=0.970] [Social Life & Identity] I actually needed this lecture when I was writing my master's thesis 😭



## Section 7 — Post-Retrieval

The lab left this step mostly empty ("merged into Generator"), noting two things worth doing:
deduplicating retrieved chunks, and optionally summarizing them before passing to the LLM if
context is too long. We implement deduplication and length-based truncation here; cross-encoder
reranking (also mentioned in the lab) is noted but not implemented, given the time constraint --
it is flagged as a future extension.

In [23]:
def postProcessRetrieval(retrieved_chunks, max_chars=2000):
    """
    retrieved_chunks: list of text strings (already ranked, best first).
    Deduplicates and joins until max_chars is reached.
    """
    seen = set()
    context_parts = []
    total_len = 0

    for chunk in retrieved_chunks:
        if chunk in seen:
            continue
        seen.add(chunk)
        if total_len + len(chunk) > max_chars:
            break
        context_parts.append(chunk)
        total_len += len(chunk)

    return "\n---\n".join(context_parts)


## Section 8 — Generator (Groq API)

**Adaptation:** the lab used DSPy `Signature` classes with a local Ollama LLM. We use the Groq
API directly with a structured prompt (question + context → answer), which is faster to set up
and needs no local model hosting. The system prompt explicitly instructs the model to answer only
from the provided context and say so if the context doesn't contain the answer -- this is the
hallucination guard the router (Section 9) relies on.

In [45]:
GROQ_API_KEY = ""

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [46]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY, timeout=20.0, max_retries=1)
MODEL_NAME = "llama-3.3-70b-versatile"

# isolate the test: skip retrieval, use a hardcoded context first
try:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": "Say hello in one word."}],
        max_tokens=10
    )
    print("API OK:", response.choices[0].message.content)
except Exception as e:
    print("API FAILED:", e)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

API OK: Hello.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

## Section 9 — Agent Orchestration & Routing

This is the hallucination-prevention design from the project plan: retrieval confidence
determines the path.

- **Confident match** (hybrid retrieval score above threshold) → answer from the RAG context
- **Low confidence** (question isn't covered by the dataset) → fall back to a live web search,
  clearly labeled as external information, not a dataset insight

This mirrors the "abstention mechanism for unanswerable questions" concept -- instead of silently
guessing, the agent explicitly routes away from the dataset when it doesn't have a good answer.

In [47]:
def ragAnswer(question, k=5, alpha=0.5, verbose=True):
    hybrid_results = hybridRetrieval(question, k=k, alpha=alpha)
    context = postProcessRetrieval([text for text, _ in hybrid_results])
    answer = generateAnswer(question, context, source_label="our YouTube comment dataset")

    if verbose:
        print(f"Q: {question}")
        print(f"A: {answer}")
        print()

    return {"answer": answer}

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

## Section 10 — Demo / Test Queries

A few known (dataset-covered) and unknown (out-of-dataset) questions to demonstrate routing.

In [48]:
test_questions = [
    "Why do people criticize influencers who promote de-influencing content?",     # known -- dataset topic
    "What do financially cautious commenters say about saving money?",             # known -- dataset topic
    "What is today's exchange rate between USD and EUR?",                          # unknown -- not in dataset
    "Who won the most recent Super Bowl?",                                         # unknown -- not in dataset
]

results = [ragAnswer(q) for q in test_questions]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Q: Why do people criticize influencers who promote de-influencing content?
A: I don't know based on the available data.



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Q: What do financially cautious commenters say about saving money?
A: Financially cautious commenters mention that savings is not encouraged due to low interest rates, and there's a risk of losing money if banks collapse. They also express skepticism about certain consumption habits, such as shopping from a paid storage unit, questioning how that can be considered saving money. Additionally, one commenter mentions choosing not to spend money on expensive appearance-related procedures, citing the high cost of botox, lasers, and fillers.



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Q: What is today's exchange rate between USD and EUR?
A: I don't know based on the available data.



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Q: Who won the most recent Super Bowl?
A: I don't know based on the available data.



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

## Section 11 — Save Artifacts for Dashboard

The FAISS index and retrieval functions are reused directly in Notebook 8 (Dashboard) so the
same RAG system powers both the Q&A demo cells above and the interactive Streamlit app.

In [49]:
# FAISS index already saved to rag_store/faiss_index in Section 5.
# Save routing config + a summary for the dashboard/report.
rag_config = {
    "embedding_model": "all-MiniLM-L6-v2",
    "vector_store": "FAISS",
    "llm_model": MODEL_NAME,
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "num_documents_indexed": len(documents),
    "retrieval_strategies": ["semantic_mmr", "lexical_tfidf", "hybrid", "metadata_filtered"],
}

with open('rag_store/rag_config.json', 'w') as f:
    json.dump(rag_config, f, indent=2)

print('Saved rag_store/faiss_index and rag_store/rag_config.json')
print(json.dumps(rag_config, indent=2))


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Saved rag_store/faiss_index and rag_store/rag_config.json
{
  "embedding_model": "all-MiniLM-L6-v2",
  "vector_store": "FAISS",
  "llm_model": "llama-3.3-70b-versatile",
  "confidence_threshold": 0.08,
  "num_documents_indexed": 80320,
  "retrieval_strategies": [
    "semantic_mmr",
    "lexical_tfidf",
    "hybrid",
    "metadata_filtered"
  ]
}


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

## Summary

Adapted the course's RAG lab sequence (Wikipedia/PDF + Ollama + Weaviate) to this project's
YouTube comment dataset: comments are treated as pre-chunked documents with contextual
prefixes (video/perspective), embedded with `sentence-transformers` instead of a local Ollama
server, and indexed in FAISS instead of Weaviate Cloud (avoiding an external account
dependency under time pressure).

Four retrieval strategies were implemented -- semantic (MMR), lexical (TF-IDF from Notebook 6),
hybrid (weighted fusion, mirroring the lab's Weaviate `alpha` blend), and metadata-filtered --
satisfying the project brief's full retrieval requirement. Generation uses the Claude API with a
context-grounded system prompt.

The **agent router** is the key hallucination-prevention component from the project design:
questions with strong retrieval matches are answered from the dataset; questions the dataset
doesn't cover are routed to a live web search instead of being guessed at, and the source is
always labeled in the answer.

**Limitations:** (1) the `CONFIDENCE_THRESHOLD` was set by quick manual testing, not a
systematic sweep -- Notebook 8 (Evaluation) should validate/tune it against a larger question
set. (2) Cross-encoder reranking (mentioned in the lab's post-retrieval step) was not implemented
due to time constraints -- noted as a future extension. (3) HyDE and multi-hop query expansion
(lab Tasks 6-7) were not implemented for the same reason; the four retrieval strategies above
already meet the project's retrieval-diversity requirement.